In [ ]:
import os
import urllib.parse
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv

# --- 1. KẾT NỐI VÀ KÉO DỮ LIỆU
load_dotenv()
db_user = os.getenv("DB_USER")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")
db_password = urllib.parse.quote_plus(os.getenv("DB_PASSWORD")) if os.getenv("DB_PASSWORD") else None

DATABASE_URL = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(DATABASE_URL)

# Kéo dữ liệu từ view
query = """
    SELECT hourly_bucket, total_energy, is_holiday, is_semester, is_exam
    FROM bi_mart.mv_bi_mart_hourly_measures
"""
print("Đang kéo dữ liệu từ Supabase...")
df = pd.read_sql_query(query, engine)

if df.empty:
    print("Không có dữ liệu!")
else:
    # --- 2. XỬ LÝ KIỂU DỮ LIỆU ---
    # Chuyển các cột text (0, 1) sang kiểu category hoặc boolean để dễ phân tích
    cols_to_convert = ['is_holiday', 'is_semester', 'is_exam']
    for col in cols_to_convert:
        df[col] = df[col].astype(str).str.strip() # Dọn dẹp khoảng trắng nếu có

    print("\n--- 3. BẢNG TỔNG HỢP (PIVOT TABLES) ---")
    
    # 3.1: So sánh tổng quan mức năng lượng trung bình giữa các trạng thái
    # Pattern: Năng lượng trung bình có giảm vào ngày nghỉ lễ không?
    holiday_impact = df.groupby('is_holiday')['total_energy'].agg(['mean', 'median', 'count']).round(2)
    print("\nSự ảnh hưởng của Ngày nghỉ lễ (is_holiday):")
    display(holiday_impact)

    # Pattern: Mùa thi (is_exam) có làm thay đổi mức năng lượng so với ngày thường không?
    exam_impact = df.groupby('is_exam')['total_energy'].agg(['mean', 'median', 'count']).round(2)
    print("\nSự ảnh hưởng của Mùa thi (is_exam):")
    display(exam_impact)

    # --- 4. TRỰC QUAN HÓA (VISUALIZATION) ĐỂ TÌM PATTERN NGẦM ĐỊNH ---
    # Thiết lập phong cách đồ thị
    sns.set_theme(style="whitegrid")
    
    # Tạo một figure chứa 2 biểu đồ
    fig, axes = plt.subplots(2, 1, figsize=(14, 12))

    # Biểu đồ 1: Đường cong năng lượng trong ngày (Hourly Profile) phân theo Học kỳ (is_semester)
    sns.lineplot(
        data=df, 
        x='hourly_bucket', 
        y='total_energy', 
        hue='is_semester', 
        estimator='mean', 
        errorbar=None, # Tắt dải sai số để đường nét rõ ràng hơn
        ax=axes[0],
        palette="Set1",
        linewidth=2.5
    )
    axes[0].set_title('Đường cong Năng lượng Trung bình theo Khung giờ: Trong Học kỳ vs Ngoài Học kỳ', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Khung giờ trong ngày (0-23)')
    axes[0].set_ylabel('Năng lượng (Total Energy)')
    axes[0].set_xticks(range(0, 24))

    # Biểu đồ 2: Đường cong năng lượng trong ngày phân theo Mùa thi (is_exam)
    sns.lineplot(
        data=df, 
        x='hourly_bucket', 
        y='total_energy', 
        hue='is_exam', 
        estimator='mean', 
        errorbar=None,
        ax=axes[1],
        palette="Set2",
        linewidth=2.5
    )
    axes[1].set_title('Đường cong Năng lượng Trung bình theo Khung giờ: Ngày thường vs Mùa Thi', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Khung giờ trong ngày (0-23)')
    axes[1].set_ylabel('Năng lượng (Total Energy)')
    axes[1].set_xticks(range(0, 24))

    plt.tight_layout()
    plt.show()